# Firm Characteristics

Investors often describe a strategy as buying cheap stocks, profitable companies or firms investing conservatively. Before these descriptions can guide an investment, they must become measurable comparisons. We need a definition of cheapness or profitability, together with a date when the investor could have observed the information.

Firm characteristics make these descriptions measurable. The definition tells us what we compare, while the timing tells us whether the information could have been used to invest. A useful comparison needs consistent units and information available before the decision. It does not tell us on its own which firms will earn higher returns.

In this notebook, I first explain how to measure firm characteristics, then apply the framework to the Fama–French construction. The application uses the repository’s locally prepared accounting and market data to follow observed records from their inputs to a dated cross-sectional comparison. [02](02_portfolio_formation.ipynb) subsequently turns such comparisons into portfolios.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from factor_asset_pricing.characteristics import fama_french_characteristics
NAVY, BLUE, TEAL, AMBER, GRAY = '#17324D', '#2878A5', '#168A86', '#D99A32', '#737D87'
plt.rcParams.update({'font.family':'DejaVu Sans', 'font.size':11,
    'axes.titlesize':12, 'axes.labelsize':11, 'xtick.labelsize':10, 'ytick.labelsize':10,
    'legend.fontsize':10, 'figure.dpi':110, 'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False, 'lines.linewidth':1.8,
    'axes.prop_cycle':plt.cycler(color=[BLUE,TEAL,AMBER,NAVY]), 'grid.alpha':.15})

from factor_asset_pricing.wrds_data import load_prepared_data
RAW_WRDS = ROOT / "data/raw/wrds"
PROCESSED = ROOT / "data/processed"


## 1. Firm Characteristics and Investment Signals

### 1.1 What characteristics measure

A firm characteristic puts a number on some aspect of a company or its security at a particular date. It can describe the scale of the business, its valuation, operating performance or recent trading history. Some inputs are observed directly, while other characteristics are ratios or estimates based on several observations. Here $x_{i,t}$ is the measurement for firm or security $i$ at date $t$.

What we compare depends on how we define the measurement. A profitability measure scaled by sales describes a different relationship from one scaled by equity. A volatility estimate depends on its return window and sampling frequency. A label alone is therefore not enough. We need to state the inputs, units, denominator and observation period.

### 1.2 From measurements to investment signals

To use a characteristic as an investment signal, we need a rule for how it guides our positions. Ranking firms by a valuation ratio turns an individual measurement into a relative comparison. Choosing to buy the high-ranked or low-ranked group is the investment hypothesis, rather than part of measuring the characteristic. Measuring the characteristic and deciding how to trade on it are two different choices.

Characteristics are also distinct from factor exposures. A characteristic describes a firm or security, while a beta describes how its returns co-move with a benchmark. A characteristic-based strategy may produce a traded factor, but its exposure and abnormal return must still be evaluated. Neither a high characteristic value nor a familiar factor label establishes a priced risk.

### 1.3 Information availability and look-ahead bias

When we evaluate a historical strategy, we can only use information an investor could have observed before choosing the holdings. A fiscal date identifies the period covered by the accounts. A publication date identifies when the accounts became accessible. A formation date identifies when the strategy uses them. These can be three different dates.

For holdings chosen before period $t$, the signal must belong to the available information set,

$$x_{i,t-1}\in\mathcal F_{t-1}.$$

Look-ahead bias enters when a historical decision uses information that only became known afterward. Accounts for a year ending in December cannot automatically be used to invest in that December. A formation lag helps separate measurement from investment, but it cannot establish actual publication dates or remove later accounting revisions. The empirical application below distinguishes the dates observed in the local annual export from its assumed availability convention.

### 1.4 Measurement choices and comparability

Before comparing firms, we need compatible units and a clear rule for which observations to include. A market value measured in thousands and an accounting value measured in millions must be aligned before forming a ratio. Missing inputs can prevent a measurement, while a missing expense treated as zero changes the numerator. A small denominator can generate a large ratio without a correspondingly large business outcome.

We need the same care with dates and identifiers. Annual growth needs two consecutive annual observations for the same company, rather than simply two adjacent rows. Company accounts and security prices also describe different entities when a company has multiple securities. In the example, market equity is measured for each security, while accounts belong to the company. The company and security identifiers are therefore kept separate.

These choices affect which firms can be compared and what the resulting numbers mean. Their consequences should be visible in the application’s coverage and distributions. Survival-based samples, missing firms and historical linking decisions remain data-preparation concerns.


## 2. Example: Fama–French Characteristics

The Fama–French construction uses size, book-to-market, operating profitability and investment to organize firms into portfolios. For this example, I use the locally prepared data to show how these characteristics are measured. The definitions make the comparison explicit. They do not establish whether associated return patterns reflect risk, mispricing, or another mechanism.

The aim is to make descriptions such as cheap, profitable, or conservative concrete. I follow the measurements from their inputs to the resulting characteristics, then examine their coverage and distributions. I leave portfolio assignments and factor performance to 02–03.

### 2.1 Data and measurement conventions

The data come from the local monthly panel and independent annual accounts prepared in [00](00_crsp_compustat_data_preparation.ipynb) from original WRDS exports. The saved inputs are checked against the original exports before use. `permno` identifies a security and `gvkey` a company. Accounting amounts are in millions of dollars, prices in dollars and shares in thousands. A month-start date labels the monthly observation. It does not mean the return was known at the start of the month.

I use June 2020 as a fixed cross-section, chosen before examining the measurements. Its starting universe contains the prepared panel’s NYSE, AMEX and Nasdaq securities with positive observed June market equity. It is a linked security universe, not a reconstruction of every officially eligible French stock. No realized-return filter is imposed to measure these characteristics.

For the June example, I use fiscal-2019 accounts directly from the cleaned Compustat table. These are the accounts assigned to holdings starting in July 2020 under the stated annual convention. Actual statement publication dates are unavailable. Investment requires consecutive fiscal years. Later calendar-year statements cannot enter an earlier formation merely because their fiscal-year label matches. See [00](00_crsp_compustat_data_preparation.ipynb) for source and timing diagnostics.

| Characteristic | Measurement | Inputs for June 2020 |
|---|---|---|
| Size | Market equity | June 2020 price and shares |
| Book-to-market | Book equity / market equity | Fiscal-2019 accounts and December 2019 market equity |
| Operating profitability | Operating surplus / book equity | Fiscal-2019 accounts |
| Investment | Annual asset growth | Fiscal-2018 and fiscal-2019 accounts |

**Size and book-to-market.** Market equity is $ME_{i,t}=|PRC_{i,t}|SHROUT_{i,t}$. CRSP’s negative price flag denotes a quote convention rather than a negative economic value. Dividing this product by 1,000 expresses market equity in millions, matching the accounting inputs. Size uses June market equity, while valuation uses the previous December denominator:

$$BE=SEQ+TXDITC-PS,\qquad BM_{i,2020}=\frac{BE_{i,2019}}{ME^{Dec}_{i,2019}}.$$

$SEQ$ is shareholders’ equity, $TXDITC$ deferred taxes and investment tax credit, and $PS$ preferred stock. Subtracting preferred stock leaves the common-equity accounting measure. The helper selects redemption value first, liquidation value second and carrying value last. Missing deferred taxes and preferred stock become zero, and nonpositive book equity becomes missing. A higher BM means more book equity per dollar of market valuation. It does not establish undervaluation.

**Profitability and investment.** The retained measurements are

$$OP_{i,2020}=\frac{REVT_{i,2019}-COGS_{i,2019}-XSGA_{i,2019}-XINT_{i,2019}}{BE_{i,2019}},\qquad
INV_{i,2020}=\frac{AT_{i,2019}-AT_{i,2018}}{AT_{i,2018}}.$$

Revenue less cost of goods sold, selling/general/administrative expenses and interest supplies the operating-surplus numerator. OP scales it by equity, not sales. Investment measures asset growth, not capital expenditure alone. A ratio of 0.10 means 10% annual growth.

For OP, the calculation requires revenue, COGS and positive book equity, while missing SG&A and interest are treated as zero. This follows the reference replication, but differs from French’s published expense-availability rule. That difference matters when comparing the resulting factors. Investment additionally requires consecutive fiscal years and positive lagged assets.


In [ ]:
FORMATION_YEAR = 2020
formation_month = pd.Timestamp(FORMATION_YEAR, 6, 1)
formation_date = pd.Timestamp(FORMATION_YEAR, 6, 30)
december_month = pd.Timestamp(FORMATION_YEAR-1, 12, 1)
raw, original_annual = load_prepared_data(PROCESSED, RAW_WRDS,
    start=f"{FORMATION_YEAR-1}-01-01", end=formation_month)
assert not raw.duplicated(["permno", "mdate"]).any()
# Independent annual inputs, not accounts recovered from a future July row.
annual = original_annual.loc[
    original_annual.fyear.isin([FORMATION_YEAR-2, FORMATION_YEAR-1]) &
    original_annual.available_from.le(pd.Timestamp(FORMATION_YEAR, 7, 1))
].copy()
assert not annual.duplicated(["gvkey", "fyear"]).any()
api_panel = raw.rename(columns={"date": "crsp_date", "mdate": "date"})
measured = fama_french_characteristics(api_panel, annual_data=annual)
formation = measured.loc[measured.date.eq(formation_month) & measured.exchcd.isin([1,2,3]) &
                          measured.me.gt(0), ['permno','gvkey','me','me_dec','bm','op','inv']].copy()
formation['size_millions'] = formation.me/1000
formation = formation.sort_values('permno').reset_index(drop=True)
assert len(formation)>0 and formation.permno.is_unique
measure_columns = ['size_millions','bm','op','inv']
print(f'June {FORMATION_YEAR}: {len(formation):,} eligible security records in the locally prepared linked universe.')
print('Independent annual records use the assumed July holding-availability convention, not verified filing dates.')


### 2.2 Computing characteristics from observed records

I use `fama_french_characteristics` to calculate the characteristics and align the accounting and market inputs. Cleaning and historical linking are done in notebook 00. The calculation here uses those prepared inputs, without establishing historical publication dates or which accounting revisions investors originally observed. The calculation uses independent annual inputs for fiscal 2018 and 2019 within the declared availability window. This also prevents growth across a missing fiscal year from being mislabeled as annual investment.

To make the calculation easier to follow, I show the first three distinct companies ordered by security identifier among June records with all four measurements. This selects examples without using future returns or choosing unusually attractive characteristic values. The two panels below show their observed accounting inputs and dated market inputs with computed outputs. Identifiers are used because this local annual export does not supply company names.


In [ ]:
complete = formation.loc[formation[measure_columns].notna().all(axis=1)]
examples = complete.drop_duplicates('gvkey').head(3).copy()
assert len(examples)==3
accounts_current = annual.loc[annual.fyear.eq(FORMATION_YEAR-1)].set_index('gvkey')
accounts_prior = annual.loc[annual.fyear.eq(FORMATION_YEAR-2)].set_index('gvkey')
worked = examples.merge(accounts_current.reset_index(),on='gvkey',how='left',validate='many_to_one')
worked = worked.merge(accounts_prior[['at','datadate']].rename(columns={'at':'prior_at',
    'datadate':'prior_datadate'}).reset_index(),on='gvkey',how='left',validate='many_to_one')
market_inputs = raw.loc[raw.mdate.eq(december_month),['permno','prc','shrout']].rename(
    columns={'prc':'dec_price','shrout':'dec_shares'})
market_inputs = market_inputs.merge(raw.loc[raw.mdate.eq(formation_month),['permno','prc','shrout']]
    .rename(columns={'prc':'june_price','shrout':'june_shares'}),on='permno',validate='one_to_one')
worked = worked.merge(market_inputs,on='permno',validate='one_to_one')
worked['preferred'] = worked.pstkrv.combine_first(worked.pstkl).combine_first(worked.pstk).fillna(0)
worked['book_equity'] = worked.seq + worked.txditc.fillna(0) - worked.preferred
worked['surplus'] = worked.revt-worked.cogs-worked.xsga.fillna(0)-worked.xint.fillna(0)
print('Observed accounting inputs: dollar amounts in millions. Missing expenses remain visible.')
account_panel = worked[['permno','gvkey','fyear','datadate','seq','txditc','preferred','revt','cogs',
                        'xsga','xint','prior_datadate','prior_at','at']].copy()
# Transposing keeps three records readable while exposing every input.
display(account_panel.set_index('permno').T)
print('Dated market inputs and measured characteristics: market equity in $m, other fields are ratios.')
result_panel = worked[['permno','gvkey','dec_price','dec_shares','june_price','june_shares',
                        'me_dec','size_millions','book_equity','bm','op','inv']].rename(
    columns={'dec_price':'December price ($)', 'dec_shares':'December shares (000s)',
             'june_price':'June price ($)', 'june_shares':'June shares (000s)',
             'me_dec':'December ME ($m)','size_millions':'June ME ($m)','book_equity':'BE ($m)',
             'bm':'BM','op':'OP','inv':'INV'})
display(result_panel.set_index('permno').round(6).T)
r = worked.iloc[0]
print(f"Security {int(r.permno)}, company {r.gvkey}: BE = {r.seq:.3f} + {r.txditc if pd.notna(r.txditc) else 0:.3f} "
      f"− {r.preferred:.3f} = {r.book_equity:.3f} million.")
print(f"BM = {r.book_equity:.3f}/{r.me_dec:.3f} = {r.bm:.6f}.")
print(f"OP = {r.surplus:.3f}/{r.book_equity:.3f} = {r.op:.6f}.")
print(f"INV = ({r['at']:.3f} − {r.prior_at:.3f})/{r.prior_at:.3f} = {r.inv:.6f} ({100*r.inv:.3f}%).")


The displayed calculation follows one observed record through the accounting adjustment, valuation denominator and two business measurements. The same rules apply to the other records. The displayed securities differ in size, measured profitability, asset growth and valuation. Read their rankings from the calculated table. The company with the greatest market value need not rank highest on another characteristic. A firm can therefore rank highly on one characteristic and lower on another. There is no single ranking of investment attractiveness here. A missing expense shown in the input table is handled by the declared helper convention, rather than silently becoming an observed zero.

June and December market values have different roles. Changing the June price changes formation size, while leaving the December denominator of this year’s BM unchanged. Similarly, high OP can arise from a relatively small equity base, so it should not be read as a sales margin. These measurements compare firms along distinct dimensions and do not establish their expected returns.

The accounts are aligned by the stated formation convention. Their actual publication dates are not available in this local annual export.


In [ ]:
# Independent formulas and dated joins verify every eligible security, not just the examples.
manual_accounts = annual.copy()
preferred_manual = np.select([manual_accounts.pstkrv.notna(), manual_accounts.pstkl.notna()],
    [manual_accounts.pstkrv.to_numpy(dtype=float,na_value=np.nan),
     manual_accounts.pstkl.to_numpy(dtype=float,na_value=np.nan)],
    default=manual_accounts.pstk.fillna(0).to_numpy(dtype=float))
manual_accounts['BE_check'] = (manual_accounts.seq + manual_accounts.txditc.fillna(0)-preferred_manual)
manual_accounts['BE_check'] = manual_accounts.BE_check.where(manual_accounts.BE_check.gt(0))
manual_current = manual_accounts.loc[manual_accounts.fyear.eq(FORMATION_YEAR-1)].copy()
manual_current = manual_current.merge(accounts_prior[['at']].rename(columns={'at':'lag_assets'}).reset_index(),
                                    on='gvkey',how='left',validate='one_to_one')
manual_current['OP_check'] = ((manual_current.revt-manual_current.cogs-manual_current.xsga.fillna(0)-
    manual_current.xint.fillna(0))/manual_current.BE_check).where(
    manual_current.revt.notna() & manual_current.cogs.notna() & manual_current.BE_check.gt(0))
manual_current['INV_check'] = ((manual_current['at']-manual_current.lag_assets)/manual_current.lag_assets).where(
    manual_current.lag_assets.gt(0))
dec_raw = raw.loc[raw.mdate.eq(december_month),['permno','prc','shrout']].copy()
dec_raw['DEC_check'] = (dec_raw.prc.abs()*dec_raw.shrout/1000).where(dec_raw.shrout.gt(0))
check = formation.merge(manual_current[['gvkey','BE_check','OP_check','INV_check']],on='gvkey',
                        how='left',validate='many_to_one').merge(dec_raw[['permno','DEC_check']],
                        on='permno',how='left',validate='one_to_one')
check['BM_check'] = (check.BE_check/check.DEC_check).where(check.DEC_check.gt(0))
for api_name, independent_name in [('bm','BM_check'),('op','OP_check'),('inv','INV_check')]:
    np.testing.assert_allclose(check[api_name].to_numpy(dtype=float,na_value=np.nan),
        check[independent_name].to_numpy(dtype=float,na_value=np.nan),rtol=1e-12,atol=1e-12,equal_nan=True)
june_raw = raw.loc[raw.mdate.eq(formation_month)].set_index('permno').loc[formation.permno]
np.testing.assert_allclose(formation.size_millions.to_numpy(dtype=float),
                          (june_raw.prc.abs()*june_raw.shrout/1000).to_numpy(dtype=float))
for name in measure_columns:
    assert np.isfinite(formation[name].dropna().to_numpy(dtype=float)).all()
print(f'Independent dated-input arithmetic agrees for all {len(formation):,} eligible security records.')


### 2.3 Measurement coverage

A valid market observation does not guarantee usable accounting characteristics. Positive book equity, a December market observation, available operating inputs and two consecutive asset observations impose different requirements. I therefore summarize each characteristic on its own available sample, while keeping the same June-universe denominator for coverage.

The table first shows how many securities we can measure, both as a count and as a share of the June universe. The median and interquartile range then show what a typical measurement looks like and how much firms differ. These summaries describe the cross-section without letting a few extreme ratios dominate an average. They are not portfolio breakpoints, and no common-complete-case filter is applied to the distributions.


In [ ]:
labels = {'size_millions':'Size ($m)','bm':'Book-to-market (ratio)',
          'op':'Operating profitability (ratio)','inv':'Investment (ratio)'}
coverage_rows = []
for name in measure_columns:
    v = formation[name].dropna().astype(float)
    coverage_rows.append({'Characteristic':labels[name], 'Usable records':len(v),
        'Coverage (%)':100*len(v)/len(formation), '25th percentile':v.quantile(.25),
        'Median':v.median(), '75th percentile':v.quantile(.75)})
coverage = pd.DataFrame(coverage_rows).set_index('Characteristic')
display(coverage.round(4))
for name, row in zip(measure_columns,coverage.itertuples(index=False)):
    v = formation[name].dropna().to_numpy(dtype=float)
    assert row[0] == np.isfinite(v).sum()
    np.testing.assert_allclose([row[2],row[3],row[4]],np.percentile(v,[25,50,75]))
print(f'{len(complete):,} securities have all four measurements. That subset selects the worked records only.')


### 2.4 Cross-sectional distributions

Before turning these measurements into portfolios, I want to see how much firms actually differ. The distributions show that variation across the same investment universe. Size is displayed on a logarithmic horizontal scale because capitalization spans orders of magnitude. The other panels retain their original ratio units, including zero and negative values where applicable.

To keep the figures readable, I display the 1st–99th percentile range separately for BM, OP and investment. These limits affect only the figure. Each panel states how many observations lie outside the displayed range, and the coverage table and calculations retain them. This is not winsorization or a change to the measured dataset.


In [ ]:
fig, axes = plt.subplots(2,2,figsize=(11,7.5),layout='constrained')
display_limits = {}
for ax, name, color in zip(axes.flat,measure_columns,[NAVY,BLUE,TEAL,AMBER]):
    v = formation[name].dropna().astype(float)
    if name=='size_millions':
        bins = np.geomspace(v.min(),v.max(),31)
        ax.hist(v,bins=bins,color=color,edgecolor='white',linewidth=.35)
        ax.set_xscale('log')
        outside = 0
    else:
        lo, hi = v.quantile([.01,.99])
        display_limits[name] = (float(lo),float(hi))
        shown = v.loc[v.between(lo,hi)]
        outside = len(v)-len(shown)
        ax.hist(shown,bins=30,range=(lo,hi),color=color,edgecolor='white',linewidth=.35)
        if lo<=0<=hi:
            ax.axvline(0,color=GRAY,lw=.9,ls='--')
    ax.set(xlabel=labels[name],ylabel='Number of security records',
           title=f'{labels[name].split(" (")[0]}: n={len(v):,}')
    ax.text(.98,.96,f'{outside} outside displayed range',transform=ax.transAxes,
            ha='right',va='top',fontsize=9,
            bbox={'facecolor':'white','edgecolor':'none','alpha':.85})
    ax.grid(axis='y',alpha=.15)
fig.suptitle(f'Prepared linked universe at June {FORMATION_YEAR}: characteristic distributions',fontsize=13)
plt.show()
for name,(lo,hi) in display_limits.items():
    v=formation[name].dropna().astype(float)
    print(f'{labels[name]}: full observed range [{v.min():.4g}, {v.max():.4g}], '
          f'displayed [{lo:.4g}, {hi:.4g}].')


*Empirical June-2020 cross-section from the locally prepared linked panel.* The vertical axes count security records rather than capital invested. The size panel therefore describes the distribution of security capitalizations, not each firm’s contribution to a value-weighted market portfolio.

In the saved June-2020 cross-section, median security market equity is about USD 632 million, while its interquartile range runs from roughly USD 136 million to USD 3.22 billion. Median BM is approximately 0.505, measured profitability 0.152 and annual investment 7.12%. The 25th percentiles of profitability and investment are negative, so losses and asset contraction are not limited to isolated extreme records. Read these summaries together with each panel. A broad size range makes the choice between equal and value weights consequential. Dispersion in BM, OP and investment supplies the comparisons needed for later sorts, but extreme ratios also make denominator checks and measurement conventions important. Negative profitability indicates a negative measured operating surplus, while negative investment indicates a contraction of reported assets. Neither observation alone establishes a return forecast.

The panels can have different sample sizes because the measurements require different inputs. Excluding unavailable values makes each comparison well defined, but it also changes which securities the comparison represents. The linked universe and its measurement coverage should therefore accompany any subsequent portfolio interpretation.


The example shows how the measurements are calculated, which firms we can measure, and what timing assumptions we use. Portfolio assignments and the interpretation of premiums come in later notebooks. [02](02_portfolio_formation.ipynb) takes the next practical step: defining breakpoints, holdings and portfolio weights from these comparisons.


## References

- Fama, E. F., and French, K. R. (2015). A five-factor asset pricing model. *Journal of Financial Economics*, 116(1), 1–22. [DOI](https://doi.org/10.1016/j.jfineco.2014.10.010).
- Kenneth French, [five-factor construction documentation](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_5_factors_2x3.html).
